# Iterators

Every type in this chapter has been an ADT you interact with through explicit operations: `Push`, `Pop`, `Get`, `Set`. `foreach` is different — it is a loop that lets a *type* decide what "give me the next element" means, without the caller ever touching an index or a key. This section covers how that contract works, and how to implement it for your own types.

```{contents}
:local:
:depth: 2
```

```{index} IEnumerable; IEnumerator
```



```{index} Why foreach Works
```

## Why foreach Works




`foreach` is not built-in magic tied to `List<T>` or `Dictionary<TKey, TValue>` specifically. It works on *any* type that implements `IEnumerable<T>` — the interface that promises "I can hand out an object that walks through my elements one at a time." That helper object implements `IEnumerator<T>`, with a `MoveNext()` method and a `Current` property.

```csharp
interface IEnumerable<T>
{
    IEnumerator<T> GetEnumerator();
}

interface IEnumerator<T>
{
    T Current { get; }
    bool MoveNext();
}
```

Written out, `foreach (var item in collection)` is shorthand for a loop built from exactly these two members:

```csharp
var e = collection.GetEnumerator();
while (e.MoveNext())
{
    var item = e.Current;
    // loop body
}
```

This is the ADT pattern from earlier sections applied to iteration itself: `IEnumerable<T>` is a contract, and any representation — an array, a linked structure, a database cursor — can satisfy it.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Producing a Sequence from Storing One
```



```{index} Separate Producing a Sequence from Storing One
```

## Separate Producing a Sequence from Storing One



A collection knows where its elements are stored. A client often needs only to visit those elements in the promised order. Enumeration provides a boundary between those responsibilities. Foreach asks for an enumeration protocol rather than directly reading private array slots or node links.

For a simple countable sequence 2, 4, 6, imagine an enumerator positioned before the first value. A successful MoveNext advances to the next value and permits reading Current. After the final value, MoveNext reports false. The client does not invent an extra zero value merely because iteration has ended. This success/result separation resembles the Try operations from earlier collection lessons.

| Request | Outcome | Current value to consume |
| --- | --- | --- |
| First MoveNext | true | 2 |
| Second MoveNext | true | 4 |
| Third MoveNext | true | 6 |
| Fourth MoveNext | false | No next element |

An IEnumerable<T> describes something that can provide enumeration. An IEnumerator<T> represents progress through one enumeration. They are not interchangeable names for a collection. Multiple enumeration requests can have separate progress states, and a producer may compute elements instead of keeping all of them in a list.




```{index} A Yield Point Suspends Rather than Finishes the Sequence
```



```{index} A Yield Point Suspends Rather than Finishes the Sequence
```

## A Yield Point Suspends Rather than Finishes the Sequence



The pattern `IEnumerable<T> Produce(...) { ... yield return value; ... }` defines an iterator. Yielding supplies one element and suspends that enumeration's progress until the consumer requests another. An ordinary return from a value-returning method finishes that call; yield return participates in a sequence of resumptions. Explain this distinction before treating yield as abbreviated list construction.

Laziness changes when work and failures occur. Obtaining an enumerable need not run every statement that will produce its values. If the consumer stops after two elements, later elements may never be computed. An expensive calculation inside an iterator is not paid simply because an enumerable variable has been declared, but repeated enumerations may repeat that work. Do not promise caching unless the implementation explicitly provides it.

The producer also needs a mutation policy. An enumeration over shared mutable storage can be affected by changes made after the enumerable is obtained. Library collections may detect invalid mutation during enumeration; custom types must state their own behavior. Returning IEnumerable<T> does not automatically create a snapshot or make private storage immutable.

Guided prediction: an iterator announces each produced number and yields 2, 4, 6. A caller breaks after consuming 4. Which production announcements should appear? Then run a second enumeration and ask whether it starts with a fresh progress state. Use the existing runnable iterator examples to check your prediction and distinguish fresh enumeration from continuing one enumerator.


```{index} yield return
```



```{index} Writing an Iterator with yield return
```

## Writing an Iterator with yield return




Implementing `IEnumerator<T>` by hand — tracking a position field, writing `MoveNext()` and `Current` yourself — is tedious for something as simple as "count from one number to another." The `yield return` statement lets the compiler generate that machinery for you. A method that contains `yield return` becomes an iterator: each call to `yield return` produces one element and pauses; the method resumes from that point the next time an element is requested.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
static IEnumerable<int> EvenNumbers(int first, int last)
{
    for (int n = first; n <= last; n++)
    {
        if (n % 2 == 0)
        {
            yield return n;
        }
    }
}

foreach (int n in EvenNumbers(1, 10))
{
    Console.Write(n + " ");
}

2 4 6 8 10 

Nothing about `EvenNumbers` mentions `IEnumerator`, `MoveNext`, or `Current` — the compiler builds a hidden class that implements all of it. The method's return type is `IEnumerable<int>`, and the body describes *what to produce*, not *how to track position*.

```{index} IEnumerable; custom implementation
```



```{index} Making Your Own Type foreach-able
```

## Making Your Own Type foreach-able




A standalone iterator *method* is useful, but sometimes the thing you want to iterate is a type you have already defined — an ADT you built, not a built-in collection. To make `foreach` work directly on an instance of your type, implement `IEnumerable<T>` and put a `yield return` inside `GetEnumerator()`:

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Collections;

var range = new NumberRange(5, 4);
foreach (int n in range)
{
    Console.Write(n + " ");
}

class NumberRange : IEnumerable<int>
{
    private readonly int start;
    private readonly int count;

    public NumberRange(int start, int count)
    {
        this.start = start;
        this.count = count;
    }

    public IEnumerator<int> GetEnumerator()
    {
        for (int i = 0; i < count; i++)
        {
            yield return start + i;
        }
    }

    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();
}

5 6 7 8 

`IEnumerable<T>` requires the generic `GetEnumerator()`. The second, non-generic `IEnumerator IEnumerable.GetEnumerator()` line is boilerplate C# requires because `IEnumerable<T>` itself extends the older, non-generic `IEnumerable` — it just forwards to the generic version above it. Once both are present, `NumberRange` works with `foreach`, LINQ, and any other API that accepts `IEnumerable<T>`, exactly like `List<T>` does.

```{index} iterator; lazy evaluation
```



```{index} Iterators Are Lazy
```

## Iterators Are Lazy




An iterator method does not run its body when you call it — it runs one step at a time, only as far as the caller actually asks for elements. Calling `EvenNumbers(1, 1000000)` returns instantly; the loop body only executes as `foreach` requests each next value. This matters for two reasons: an iterator can describe an infinite or unbounded sequence without ever running forever, and a `foreach` that stops early (with `break`) never computes the elements it never asked for.

```{index} Practice
```

## Practice




1. Write an iterator method `static IEnumerable<int> CountDown(int from)` that yields `from, from - 1, ..., 1`.
2. Write a class `Pair<T>` holding two values of type `T`, implementing `IEnumerable<T>` so that `foreach` over a `Pair<T>` yields both values in order.
3. Explain, in terms of the `IEnumerable<T>` / `IEnumerator<T>` contract from earlier in this section, why a type can be `foreach`-able without being indexable — that is, without supporting `range[i]`.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Lists: Chapter Introduction](https://opendsa.org/OpenDSA/Books/Catalog/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
